# Part 1 — Data Generation
## Distance-3 Rotated Surface Code with Stim

**Key design principle:** Both Part 1 and Part 2 come from a **single sampling run**.
The `m2d_converter` transforms the same raw measurements into detection events,
so both decoders share identical labels — enabling a fair comparison.

```
circuit.compile_sampler()
        │
        ▼
  raw_measurements (N, 81)
        │
        ├─── [:, :72]  syndrome cols only ──────►  Part 2 input  (N, 72)
        │
        ▼  m2d_converter.convert()
  detection_events ────────────────────────────►  Part 1 input  (N, 72)
  observable_flips ────────────────────────────►  SHARED labels (N, 1)
```

**Why slice cols 0-71 only?**
The last 9 columns (72-80) are the **final data qubit readouts**.
For a `memory_z` circuit, XOR(col72, col73, col74) == observable 100% of the time —
the answer is literally in those columns. Keeping them makes Part 2 trivially solvable.
We keep only the **syndrome measurement columns** (the actual decoding problem).

### Install
```bash
pip install stim numpy
```

In [1]:
import stim
import numpy as np
import os

print(f"Stim  : {stim.__version__}")
print(f"NumPy : {np.__version__}")

Stim  : 1.15.0
NumPy : 2.2.6


---
## 1. Build the Circuit

In [2]:
NOISE    = 0.001
DISTANCE = 3
ROUNDS   = 9

circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=ROUNDS,
    distance=DISTANCE,
    after_clifford_depolarization=NOISE,
    after_reset_flip_probability=NOISE,
    before_measure_flip_probability=NOISE,
    before_round_data_depolarization=NOISE,
)

print("Circuit summary")
print(f"  Qubits       : {circuit.num_qubits}")
print(f"  Measurements : {circuit.num_measurements}   ← raw_measurements width")
print(f"  Detectors    : {circuit.num_detectors}    ← detection_events width")
print(f"  Observables  : {circuit.num_observables}     ← observable_flips width")

Circuit summary
  Qubits       : 26
  Measurements : 81   ← raw_measurements width
  Detectors    : 72    ← detection_events width
  Observables  : 1     ← observable_flips width


---
## 2. Single Sampling Run → Both Representations

This is the critical fix: one run produces inputs for both Part 1 and Part 2
with **identical labels**.

In [3]:
NUM_SHOTS = 100_000

# ── Step 1: sample raw measurements ──────────────────────────────────────────
sampler          = circuit.compile_sampler()
raw_all          = np.array(circuit.compile_sampler().sample(shots=NUM_SHOTS), dtype=np.bool_)

# ── Step 2: convert to detection events + observable flips ───────────────────
converter = circuit.compile_m2d_converter()
detection_events, observable_flips = converter.convert(
    measurements=raw_all,
    separate_observables=True,
)
detection_events = np.array(detection_events, dtype=np.bool_)
observable_flips = np.array(observable_flips, dtype=np.bool_)

# ── Step 3: slice raw to syndrome columns only ────────────────────────────────
# The full raw array has shape (N, 81):
#   cols  0–71 → ancilla/syndrome measurements  (9 rounds × 8 stabilisers)
#   cols 72–80 → FINAL DATA QUBIT READOUT       (9 data qubits)
#
# CRITICAL: XOR(col72, col73, col74) == observable_flip 100% of the time.
# The answer is in the final readout columns — keeping them makes Part 2
# trivial (the network just learns a 3-input XOR, not a decoder).
# We keep ONLY the syndrome columns so Part 2 is a genuine decoding task.
num_syndrome_cols = circuit.num_detectors   # 72 — same width as detection_events
raw_measurements  = raw_all[:, :num_syndrome_cols]   # (N, 72) — syndrome only

# ── Confirm shapes ────────────────────────────────────────────────────────────
print("=== Single sampling run output ===")
print(f"  raw_all          : {raw_all.shape}    (full circuit output, do NOT use directly)")
print(f"  raw_measurements : {raw_measurements.shape}    ← Part 2 input  (syndrome cols only)")
print(f"  detection_events : {detection_events.shape}    ← Part 1 input")
print(f"  observable_flips : {observable_flips.shape}     ← shared labels")
print()
print(f"  Logical error rate : {observable_flips.mean():.4f}  ({observable_flips.sum():,} errors)")
print()
print("Part 1 vs Part 2 now differ only in representation:")
print("  Part 1: XOR-processed detection events  (Stim does the XOR for you)")
print("  Part 2: raw syndrome bits               (network must learn the XOR structure)")

=== Single sampling run output ===
  raw_all          : (100000, 81)    (full circuit output, do NOT use directly)
  raw_measurements : (100000, 72)    ← Part 2 input  (syndrome cols only)
  detection_events : (100000, 72)    ← Part 1 input
  observable_flips : (100000, 1)     ← shared labels

  Logical error rate : 0.0549  (5,487 errors)

Part 1 vs Part 2 now differ only in representation:
  Part 1: XOR-processed detection events  (Stim does the XOR for you)
  Part 2: raw syndrome bits               (network must learn the XOR structure)


---
## 3. Sanity Checks

In [4]:
labels = observable_flips[:, 0]

# ── Class balance ─────────────────────────────────────────────────────────────
print("Class balance")
print(f"  No logical error : {(~labels).sum():,}  ({100*(~labels).mean():.2f}%)")
print(f"  Logical error    : {labels.sum():,}    ({100*labels.mean():.4f}%)")
print()

# ── Detection event statistics ────────────────────────────────────────────────
events_per_shot = detection_events.sum(axis=1)
print("Detection events per shot")
print(f"  Mean   : {events_per_shot.mean():.3f}")
print(f"  Median : {np.median(events_per_shot):.0f}")
print(f"  Max    : {events_per_shot.max()}")
print()

# ── Error rate conditioned on #events — the key insight ──────────────────────
print("Logical error rate conditioned on #detection events")
print(f"  {'#Events':<10} {'Shots':>8} {'Errors':>8} {'Error Rate':>12}")
print("  " + "-" * 42)
for k in range(0, 9):
    mask = events_per_shot == k
    if mask.sum() < 10:
        continue
    rate = labels[mask].mean()
    bar  = '█' * int(rate * 30)
    print(f"  {k:<10} {mask.sum():>8,} {labels[mask].sum():>8,} {100*rate:>10.2f}%  {bar}")

print()

# ── Raw measurement structure ─────────────────────────────────────────────────
rates = raw_measurements.mean(axis=0)
ancilla_cols = (rates > 0.4).sum()
data_cols    = (rates < 0.05).sum()
print("Raw measurement column structure (syndrome cols only, 0-71)")
print(f"  ~50% rate (ancilla qubits)   : {ancilla_cols} columns")
print(f"  ~0%  rate (data qubits reset): {data_cols} columns")
print(f"  Total                        : {raw_measurements.shape[1]} columns")
print()
print(f"  Excluded: cols 72-80 (final data qubit readout — contains the answer)")

Class balance
  No logical error : 94,513  (94.51%)
  Logical error    : 5,487    (5.4870%)

Detection events per shot
  Mean   : 0.956
  Median : 0
  Max    : 14

Logical error rate conditioned on #detection events
  #Events       Shots   Errors   Error Rate
  ------------------------------------------
  0            61,288        0       0.00%  
  1             7,607    1,777      23.36%  ███████
  2            17,820    1,151       6.46%  █
  3             5,903    1,458      24.70%  ███████
  4             4,488      454      10.12%  ███
  5             1,487      364      24.48%  ███████
  6               893      147      16.46%  ████
  7               300       94      31.33%  █████████
  8               140       26      18.57%  █████

Raw measurement column structure (syndrome cols only, 0-71)
  ~50% rate (ancilla qubits)   : 36 columns
  ~0%  rate (data qubits reset): 28 columns
  Total                        : 72 columns

  Excluded: cols 72-80 (final data qubit readout — co

---
## 4. Save to Disk

In [5]:
os.makedirs("data", exist_ok=True)

np.save("data/detection_events.npy",  detection_events)   # Part 1 input  (N, 72)
np.save("data/raw_measurements.npy",  raw_measurements)   # Part 2 input  (N, 72) syndrome only
np.save("data/observable_flips.npy",  observable_flips)   # shared labels (N, 1)

np.save("data/metadata.npy", {
    'noise': NOISE, 'distance': DISTANCE, 'rounds': ROUNDS,
    'num_shots': NUM_SHOTS,
    'num_detectors': circuit.num_detectors,
    'num_measurements': circuit.num_measurements,
    'num_syndrome_cols': num_syndrome_cols,
})

print("Saved to ./data/")
for fname in ['detection_events.npy', 'raw_measurements.npy', 'observable_flips.npy']:
    arr = np.load(f"data/{fname}")
    print(f"  {fname:<30} {str(arr.shape):<22} {arr.dtype}  ({arr.nbytes/1024**2:.2f} MB)")
print()
print("Both arrays have the same width (72) but different representations:")
print("  detection_events : XOR of consecutive syndrome rounds (processed)")
print("  raw_measurements : raw syndrome bits, no XOR applied  (unprocessed)")

Saved to ./data/
  detection_events.npy           (100000, 72)           bool  (6.87 MB)
  raw_measurements.npy           (100000, 72)           bool  (6.87 MB)
  observable_flips.npy           (100000, 1)            bool  (0.10 MB)

Both arrays have the same width (72) but different representations:
  detection_events : XOR of consecutive syndrome rounds (processed)
  raw_measurements : raw syndrome bits, no XOR applied  (unprocessed)


---
## 5. Multi-Noise Dataset for Threshold Study

Generate data at several noise levels.
Used in notebook 03 (noise sweep) to study how decoders behave near threshold.

In [6]:
NOISE_LEVELS  = [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02]
SWEEP_SHOTS   = 50_000    # per noise level

os.makedirs("data/sweep", exist_ok=True)

print(f"Generating sweep data ({SWEEP_SHOTS:,} shots × {len(NOISE_LEVELS)} noise levels)...")
print()

for p in NOISE_LEVELS:
    circ = stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=ROUNDS,
        distance=DISTANCE,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p,
    )

    raw_full = np.array(circ.compile_sampler().sample(shots=SWEEP_SHOTS), dtype=np.bool_)
    conv = circ.compile_m2d_converter()
    det, obs = conv.convert(measurements=raw_full, separate_observables=True)
    det = np.array(det, dtype=np.bool_)
    obs = np.array(obs, dtype=np.bool_)
    raw = raw_full[:, :circ.num_detectors]   # syndrome cols only — exclude final readout

    tag = f"{int(p*10000):04d}"   # e.g. 0010 for p=0.001
    np.save(f"data/sweep/raw_p{tag}.npy",  raw)
    np.save(f"data/sweep/det_p{tag}.npy",  det)
    np.save(f"data/sweep/obs_p{tag}.npy",  obs)

    print(f"  p={p:.4f}  LER={obs.mean():.5f}  saved → sweep/*_p{tag}.npy")

print()
print("Done. Sweep data ready for notebook 03.")

Generating sweep data (50,000 shots × 6 noise levels)...

  p=0.0005  LER=0.02862  saved → sweep/*_p0005.npy
  p=0.0010  LER=0.05272  saved → sweep/*_p0010.npy
  p=0.0020  LER=0.10162  saved → sweep/*_p0020.npy
  p=0.0050  LER=0.22114  saved → sweep/*_p0050.npy
  p=0.0100  LER=0.34362  saved → sweep/*_p0100.npy
  p=0.0200  LER=0.45250  saved → sweep/*_p0200.npy

Done. Sweep data ready for notebook 03.
